In [1]:
# phospholingo required
import sys
from pathlib import Path

phospholingo_dir = Path("src/files/PhosphoLingo/phospholingo").resolve()
sys.path.insert(0, str(phospholingo_dir))

import predict

/Users/newuser/anaconda3/envs/1433predictor2026/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/newuser/anaconda3/envs/1433predictor2026/lib/python3.9/site-packages/pytorch_lightning/utilities/imports.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## Feature extraction functions

In [2]:
import pandas as pd
import numpy as np

### Extract idr and achor scores functions (this function have length limitation)

In [3]:
import src.files.iupred2a.iupred2a_lib as iupred2a_lib
def extract_idr_anchor_score(sequence, site):
    iupred_type = "long"
    # Predict disorder using iupred2a_lib
    iupred_scores = iupred2a_lib.iupred(sequence, iupred_type)[0]

    # Predict anchor regions using iupred2a_lib if anchor is enabled
    anchor_scores = iupred2a_lib.anchor2(sequence)[0]
    # print(anchor_scores)

    # Prepare the data for saving into a CSV file
    data = {
        "position": list(range(1, len(sequence) + 1)),
        "amino_acid": list(sequence),
        "iupred_score": iupred_scores,
        "anchor_score": anchor_scores,
    }

    # Create a DataFrame from the data
    df = pd.DataFrame(data)
    # print(df)
    df["position"] = df["position"].astype(int)
    
    idr_score_df = df[df["position"] == int(site)][["iupred_score", "anchor_score"]]
        
    return idr_score_df

In [4]:
anchor_test_seq = "SREFAKERERVENRRAFLKLRRQQQIERELNGYLEWICKAEEVMLAEEDKNAEEKSPLDGVYINNGRIDEGTDTFETNQGSSRNIILKRATTIKKSKNDLIHAEEGEDHFTDISSVGSPFARASLKSGKNESSSYFRRKEKRFRFFIRRMVKAQSFYWIVLCIVALNTLCVAIVHYDQPQWLTDALYFAEFVFLGLFLTEMFLKMYGLGPRNYFHSSFNCFDFGVIVGSIFEVIWAAVKPGASFGISVLRALRLLRIFKVTKYWNSLRNLVVSLLNSMKSIISLLFLLFLFIVVFALLGMQLFGGQFNFEDETPTTNFDTFPAAILTVFQILTGEDWNAVMYHGIESQGGVRSGMFSSIYFIVLTLFGNYTLLNVFLAIAVDNLANAQELTKDEEEMEEATNQKLARQKAMEVAEVSPISAANISIAAKEQQKSLKSMSVWEQRTNQIRLQNLRASCEALYNELDPEERMRIASSLHIRPDMNTHLDRPLVVEKPKGDVRNNVGKTGPADGQDSEQDRLVQPGSSEAPRKHHRHRDKLNEQEKAVETNENGESGTNDKEEKRRQHQSRSKEMEGKSEGKEGKSERSRSREGGKKHHHQGVGPVEERSEKEPKQCRSHRHSTERQGKEGNGTVNGTKTERRSRHRGGSRSGNREGDPRGENGDEHPRRQKSRRKALSTYVAEDKKENGDHKNGETGEKEHKNHRPKESQPANDMGTKGLTAAPPPPTGCLKQGAEQPEDADNQKNVKRMTQPVAKTTTANIPVTITAPPGETIVIPMNNIDFESSTKKEEKKELEDDLTRNGPRQILPYSSMFVFSPTNPIRKLCHYIVSLRYFEMCILLVIAMSSIALAAEDPVQADSPWNNVLKYLDYVFTGVFTFEMVIKMIDLGLLLHPGSYFRDLWNILDFVVVSGALVAFAFSGTKGKDINTIKSLRVLRVLRPLKTIKRLPKLKAVFDCVVNSLKNVLNILIVYMLFMFIFAVVAVQLFKGKFFYCTDESKELERDCRGKYLEYEKDEVEARPREWQKYEFHYDNVLWALLTLFTVSTGEGWPTVLKHSVDATYEDQGPSPGFRMEMSIFYVVYFVVFPFFFVNIFVALIIITFQEQGDKVMSDCSLEKNERACIDFAISAKPLTRYMPQNKQSFQYKMWKFVVSPPFEYFIMAMIALNTIVLMMKFYDAPRVYEDMLKCLNIVFTSMFSMECVLKIIAFGVLNYFRDAWNIFDFVTVLGSITDILVTEIADLNNFINLSFLRLFRAARLIKLLRQGYTIRILLWTFVQSFKALPYVCLLIAMLFFIYAIIGMQVFGNIALDDETSINRHNNFRTFLQALMLLFRSATGEAWHEIMLSCLGNKPCDPRSGTSANECGNDFAYFYFVSFIFLCSFLMLNLFVAVIMDNFEYLTRDSSILGPHHLDEFIRVWAEYDPAACGRIHYKDMYNLLRVISPPLGLGKKCPNRVAYKRLVRMNMPIAEDNSVHFTSTLMALIRTALEIKLASGGPNQQQCDAELKKEIATVWPNLSQKTLDLLVPPHRPNELTVGKVYAVLMIFDYYKQNKTKKIQQQQQQQGHPQTRTTQLFQRLVTPTQEQLALLQFNNTKTVMPQQSTTSLNNGGNLTIQDIGVRDPTSCVSQLPQEVFHDSTKKAIERGHSEEIPSRQAVEMKEICPSTTNGDHQPALESQGRAASMPRLAVETQRSKARSPGSYQAPIPDTSPMKRSISSLTPQRPQGMLLHDYALEPAVPERPHHHHNHRCHRRREKKQRSLERSPSRHADRDTGQVSDLITMAPGDASSRERKHERGRSQERKHHSSSAEKQRYYsCDRYSSRDHGQPKSVIQSRSTSPNGGQDPGHPRQGSGSVNGSPATSTSGTSTPTGRGRRQLPQTPLTPRPNITFKTANSSPVQFASAHSGLPTFSPRRLSRGLSEHNALLRYESENHSHIMVTRIGSDPYLGHREDCDSSYHILLEDTLTFEEAVATNSGRSSRTSYVSSLTSQPHHTHRIPNGYHYSLGVSMGPGTGTREHQYYHEADEDDWC"
anchor_test_site = 1812

In [5]:
extract_idr_anchor_score(anchor_test_seq, anchor_test_site)

,iupred_score,anchor_score
1811,0.694846,0.384875


In [6]:
import psutil, os
print(psutil.Process(os.getpid()).memory_info().rss / 1024**3, "GB")

0.281585693359375 GB


### Extract compactness score for IDR functions

In [7]:
#@title <b>Preliminary operations</b>
import subprocess
# subprocess.run( 'pip install wget localcider==0.1.18'.split() )
# subprocess.run('pip uninstall scikit-learn -y'.split())
# subprocess.run('pip install scikit-learn==1.0.2'.split())
import numpy as np
import itertools
from localcider.sequenceParameters import SequenceParameters
import wget
import sys
import os
from joblib import dump, load
import pandas as pd
# from google.colab import files
# from ipywidgets import IntProgress
from IPython.display import display
from IPython.display import clear_output

def calc_seq_prop(seq,residues,Nc,Cc,Hc):
    seq = list(seq).copy()
    fasta_kappa = np.array(seq.copy())
    N = len(seq)
    r = residues.copy()

    # calculate properties that do not depend on charges
    fK = sum([seq.count(a) for a in ['K']])/N
    fR = sum([seq.count(a) for a in ['R']])/N
    fE = sum([seq.count(a) for a in ['E']])/N
    fD = sum([seq.count(a) for a in ['D']])/N
    faro = sum([seq.count(a) for a in ['W','Y','F']])/N
    mean_lambda = np.mean(r.loc[seq].lambdas)
    
    pairs = np.array(list(itertools.combinations(seq,2)))
    pairs_indices = np.array(list(itertools.combinations(range(N),2)))
    # calculate sequence separations
    ij_dist = np.diff(pairs_indices,axis=1).flatten().astype(float)
    # calculate lambda sums
    ll = r.lambdas.loc[pairs[:,0]].values+r.lambdas.loc[pairs[:,1]].values
    # calculate SHD
    beta = -1
    shd = np.sum(ll*np.power(np.abs(ij_dist),beta))/N

    # fix charges
    if Nc == 1:
        r.loc['X'] = r.loc[seq[0]]
        r.loc['X','q'] = r.loc[seq[0],'q'] + 1.
        seq[0] = 'X'
        if r.loc['X','q'] > 0:
            fasta_kappa[0] = 'K'
        else:
            fasta_kappa[0] = 'A'
    if Cc == 1:
        r.loc['Z'] = r.loc[seq[-1]]
        r.loc['Z','q'] = r.loc[seq[-1],'q'] - 1.
        seq[-1] = 'Z'
        if r.loc['Z','q'] < 0:
            fasta_kappa[-1] = 'D'
        else:
            fasta_kappa[-1] = 'A'
    if Hc < 0.5:
        r.loc['H', 'q'] = 0
        fasta_kappa[np.where(np.array(seq) == 'H')[0]] = 'A'
    elif Hc >= 0.5:
        r.loc['H', 'q'] = 1
        fasta_kappa[np.where(np.array(seq) == 'H')[0]] = 'K'

    # calculate properties that depend on charges
    pairs = np.array(list(itertools.combinations(seq,2)))
    # calculate charge products
    qq = r.q.loc[pairs[:,0]].values*r.q.loc[pairs[:,1]].values
    # calculate SCD
    scd = np.sum(qq*np.sqrt(ij_dist))/N
    SeqOb = SequenceParameters(''.join(fasta_kappa))
    kappa = SeqOb.get_kappa()
    fcr = r.q.loc[seq].abs().mean()
    ncpr = r.q.loc[seq].mean()
    
    return pd.Series(data=[fK,fR,fE,fD,faro,scd,shd,kappa,fcr,mean_lambda,ncpr],
                 index=['fK','fR','fE','fD','faro','SCD','SHD','kappa','FCR','mean_lambda','NCPR'])

def extract_compactness_score(idr_seq):
    aa = ['A','C','D','E','F','G','H','I','K','L','M','N','P','Q','R','S','T','V','W','Y']

    url = 'https://github.com/KULL-Centre/_2023_Tesei_IDRome/blob/main'

    if os.path.exists('model/svr_model_nu.joblib') == False:
        wget.download(url+'/svr_models/svr_model_nu.joblib?raw=true', out='model/svr_model_nu.joblib')
    if os.path.exists('model/svr_model_SPR.joblib') == False:
        wget.download(url+'/svr_models/svr_model_SPR.joblib?raw=true', out='model/svr_model_SPR.joblib')
    if os.path.exists('model/residues.csv') == False:
        wget.download(url+'/md_simulations/data/residues.csv?raw=true', out='model/residues.csv')

    model_nu = load('model/svr_model_nu.joblib') 
    model_spr = load('model/svr_model_SPR.joblib') 
    features_nu = ['SCD','SHD','kappa','FCR','mean_lambda']
    features_spr = ['SCD','SHD','mean_lambda']

    residues = pd.read_csv('model/residues.csv')
    residues = residues.set_index('one')

    fasta_dict = {}
    df = pd.DataFrame(columns=['nuSVR','SconfSVR/N (kB)','mean_lambda','SHD','SCD','kappa','FCR','NCPR',
                               'fK','fR','fE','fD','faro'])
    
    
    # Define the FASTA format: start with the header, then the sequence
    fasta_dict = {}
    protein_name = "protein1"  # You can specify any name, here I use "protein1"

    # Convert sequence string to FASTA format
    fasta_dict[protein_name] = idr_seq
    current_upload = [protein_name]

    # Validate the sequence
    for x in list(current_upload):  # Create a copy for safe modification
        valid = True
        for a in fasta_dict[x]:
            if a not in aa:
                print(f'WARNING: {x} sequence contains a character ({a}) not recognized as an amino acid. This sequence will be ignored.')
                del fasta_dict[x]
                valid = False
                break
        if not valid:
            current_upload.remove(x)

    # Output the results
    print("Valid sequences uploaded in FASTA format:")
    for name in fasta_dict:
        print(f">{name}")
        print(fasta_dict[name])
        
    #@title <b>Define charge states</b>
    #@markdown Define charge states:
    charged_N_terminal_amine = False #@param {type:"boolean"}
    charged_C_terminal_carboxyl = True #@param {type:"boolean"}
    charged_histidine = False #@param {type:"boolean"}
    Nc = 1 if charged_N_terminal_amine == True else 0
    Cc = 1 if charged_C_terminal_carboxyl == True else 0
    if charged_histidine == False:
        Hc = 0
        
    #@title <b>Predict $\nu$ and $S_\text{conf}/N$
    #@markdown Use this cell to calculate sequence features and predict the scaling exponent, $\nu$, and the conformational entropy per residue, $S_\text{conf}/N$. Results will be downloaded in a csv file.

    # f = IntProgress(min=0, max=len(fasta_dict), description='Progress:', bar_style='warning')
    # display(f)

    for k in fasta_dict.keys():
        res = calc_seq_prop(fasta_dict[k],residues,Nc,Cc,Hc)
        nu = np.around(model_nu.predict(res.loc[features_nu].values.reshape(1, -1))[0],3)
        spr = np.around(model_spr.predict(res.loc[features_spr].values.reshape(1, -1))[0],3)
        df.loc[k,'nuSVR'] = nu
        df.loc[k,'SconfSVR/N (kB)'] = spr
        df.loc[k,res.index.values] = np.around(res.loc[res.index.values].values,3)
        # f.value += 1

    clear_output()
#     df.to_csv('svr_pred.csv',index_label='name')
    
    return df

In [8]:
# test
idr_seq = "MSSQSHPDGLSGRPVELLNPARVNHMPSTVDVATALPLQVAPSAVPMDLRLDHQFSLPVAEPALREQQLQQELLALKQKQQIQRQILIAEFQRQHEQLSRQHEAQLHEHIKQQQEMLAMKHQQELLEHQRKLERHRQEQELEKQHREQKLQQLKNKEKGKESAVASTEVKMKLQEFVLNKKKALAHRNLNHCISSDPRYWYGKTQHSSLDQSSPPQSGVSTSYNHPVLGMYDAKDDFPLRKTASEPNLKLRSRLKQKVAERRSSPLLRRKDGPVVTALKKRPLDVTDSACSSAPGSGPSSPNNSSGSVSAENGIAPAVPSIPAETSLAHRLVAREGSAAPLPLYTSPSLPNITLGLPATGPSAGTAGQQDAERLTLPALQQRLSLFPGTHLTPYLSTSPLERDGGAAHSPLLQHMVLLEQPPAQAPLVTGLGALPLHAQSLVGADRVSPSIHKLRQHRPLGRTQSAPLPQNAQALQHLVIQQQHQQFLEKHKQQFQQQQLQMNKIIPKPSEPARQPESHPEETEEELREHQALLDEPYLDRLPGQKEAHAQAGVQVKQEPIESDEEEAEPPREVEPGQRQPSEQELLFRQQALLLEQQRIHQLRNYQASMEAAGIPVSFGGHRPLSRAQSSPASATFPVSVQEPPTKPRFTTGLVYDTLMLKHQCTCGSSSSHPEHAGRIQSIWSRLQETGLRGKCECIRGRKATLEELQTVHSEAHTLLYGTNPLNRQKLDSKKLLGSLASVFVRLPCGGVGVDSDTIWNEVHSAGAARLAVGCVVELVFKVATGELKNGFAVVRPPGHHAEESTPMGFCYFNSVAVAAKLLQQRLSVSKILIVDWDVHHGNGTQQAFYSDPSVLYMSLHRYDDGNFFPGSGAPDEVGTGPGVGFNVNMAFTGGLDPPMGDAEYLAAFRTVVMPIASEFAPDVVLVSSGFDAVEGHPTPLGGYNLSARCFGYLTKQLMGLAGGRIVLALEGGHDLTAICDASEACVSALLGNELDPLPEKVLQQRPNANAVRSMEKVMEIHSKYWRCLQRTTSTAGRSLIEAQTCENEEAETVTAMASLSVGVKPAEKRPDEEPMEEEPPL"
extract_compactness_score(idr_seq)

,nuSVR,SconfSVR/N (kB),mean_lambda,SHD,SCD,kappa,FCR,NCPR,fK,fR,fE,fD,faro
protein1,0.486,9.91,0.426,5.601,-3.178,0.192,0.213,-0.01,0.045,0.056,0.078,0.033,0.044


In [9]:
data = {
    'nuSVR': 0.0,
    'SconfSVR/N (kB)': 0.0,
    'mean_lambda': 0.0,
    'SHD': 0.0,
    'SCD': 0.0,
    'kappa': 0.0,
    'FCR': 0.0,
    'NCPR': 0.0,
    'fK': 0.0,
    'fR': 0.0,
    'fE': 0.0,
    'fD': 0.0,
    'faro': 0.0
}
compactness_score_df = pd.DataFrame([data], index=['protein1'])
compactness_score_df

,nuSVR,SconfSVR/N (kB),mean_lambda,SHD,SCD,kappa,FCR,NCPR,fK,fR,fE,fD,faro
protein1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [10]:
import psutil, os
print(psutil.Process(os.getpid()).memory_info().rss / 1024**3, "GB")

0.4051055908203125 GB


In [11]:
import os
print(os.system("ulimit -a"))

core file size          (blocks, -c) 0
data seg size           (kbytes, -d) unlimited
file size               (blocks, -f) unlimited
max locked memory       (kbytes, -l) unlimited
max memory size         (kbytes, -m) unlimited
open files                      (-n) 4864
pipe size            (512 bytes, -p) 1
stack size              (kbytes, -s) 8192
cpu time               (seconds, -t) unlimited
max user processes              (-u) 11136
virtual memory          (kbytes, -v) unlimited
0


In [12]:
import psutil
print(psutil.virtual_memory())

svmem(total=68719476736, available=38956523520, percent=43.3, used=25609199616, free=18185359360, active=20774592512, inactive=20531429376, wired=4834607104)


In [13]:
import resource

soft, hard = resource.getrlimit(resource.RLIMIT_AS)
print("Soft limit:", soft)
print("Hard limit:", hard)

Soft limit: 9223372036854775807
Hard limit: 9223372036854775807


In [14]:
import torch.multiprocessing as mp
print(mp.get_start_method())

spawn


In [15]:
import torch
import pytorch_lightning as pl
import sys

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Lightning:", pl.__version__)

Python: 3.9.21 | packaged by conda-forge | (main, Dec  5 2024, 13:50:36) 
[Clang 18.1.8 ]
Torch: 1.13.1
Lightning: 1.7.7


In [16]:
import gc

# Force garbage collection
gc.collect()

0

In [17]:
import psutil, os
print(psutil.Process(os.getpid()).memory_info().rss / 1024**3, "GB")

0.4051551818847656 GB


### Extract phospholingo score functions

In [18]:
import argparse
import sys
import os
import uuid
import pandas as pd
from pathlib import Path
from typing import List, Optional

# # Get the directory containing this script
# SCRIPT_DIR = Path(__file__).resolve().parent

# # 1. Add PhosphoLingo to Python path
# PHOSPHOLINGO_DIR = SCRIPT_DIR / "src/files/PhosphoLingo/phosphoLingo"  # Adjust relative path as needed
# sys.path.insert(0, str(PHOSPHOLINGO_DIR))


# # 1. Add PhosphoLingo to Python path
phospholingo_dir = Path("src/files/PhosphoLingo/phospholingo").resolve()
sys.path.insert(0, str(phospholingo_dir))

# 2. Import AFTER path adjustment
import predict

def add_at_after_st(seq: str, position: int) -> Optional[str]:
    """
    Add '@' after specified position if residue is S/T with proper error handling
    
    Args:
        seq: Input protein sequence
        position: 1-based position to modify
        
    Returns:
        Modified sequence or None if invalid
    """
    try:
        if position < 1 or position > len(seq):
            raise ValueError(f"Position {position} out of range (1-{len(seq)})")
            
        residue = seq[position-1]
        if residue not in ['s', 't']:
            raise ValueError(f"Residue {residue} at position {position} is not S/T")
            
        return seq[:position] + "@" + seq[position:]
        
    except (IndexError, ValueError) as e:
        error_msg = f"Position: {position}, Sequence: {seq[:30]}{'...' if len(seq)>30 else ''}, Error: {str(e)}"
        with open("processing_errors.log", "a") as f:
            f.write(error_msg + "\n")
        return None

def create_fasta_file(sequence_ids: List[str], modified_sequences: List[str], output_path: Path) -> None:
    """
    Create FASTA file from modified sequences
    
    Args:
        sequence_ids: List of sequence identifiers
        modified_sequences: List of modified sequences
        output_path: Path to output FASTA file
    """
    with open(output_path, 'w') as fasta_file:
        for seq_id, seq in zip(sequence_ids, modified_sequences):
            if seq is not None:
                fasta_file.write(f">{seq_id}\n{seq}\n")

def predict_fasta_file(input_fasta: Path, output_csv: Path, model_loc: str) -> None:
    """
    Run PhosphoLingo prediction on a FASTA file
    
    Args:
        input_fasta: Path to input FASTA file
        output_csv: Path to output predictions CSV
        model_loc: Path to PhosphoLingo model
    """
    # Ensure output directory exists
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    
    # Run prediction
    predict.run_predict(
        str(Path(model_loc).resolve()),
        str(input_fasta),
        str(output_csv)
    )

def extract_phospholingo_score(sequences, sites, model_loc, sequence_ids = ["default"]):
    """
    Batch process multiple sequences and extract phosphorylation scores
    
    Args:
        sequence_ids: List of unique sequence identifiers
        sequences: List of protein sequences
        sites: List of 1-based positions to predict
        model_loc: Path to PhosphoLingo model
        
    Returns:
        DataFrame with predictions and original data
    """
    # Validate input
    if len(sequence_ids) != len(sequences) or len(sequences) != len(sites):
        raise ValueError("All input lists must have the same length")
        
    # Generate modified sequences
    modified_sequences = []
    valid_entries = []
    for seq_id, seq, site in zip(sequence_ids, sequences, sites):
        modified = add_at_after_st(seq, site)
        print(modified)
        if modified:
            modified_sequences.append(modified)
            valid_entries.append((seq_id, seq, site))
            
    if not modified_sequences:
        raise ValueError("No valid sequences to process")
        
    # Create temporary files
    temp_id = uuid.uuid4().hex
    temp_fasta = Path(f"temp_{temp_id}.fasta")
    temp_output = Path(f"temp_{temp_id}_predictions.csv")
    
    try:
        # Create FASTA file
        create_fasta_file(
            [e[0] for e in valid_entries],
            modified_sequences,
            temp_fasta)
        
        # Run predictions
        predict_fasta_file(temp_fasta, temp_output, model_loc)
        
        # Load and process results
        results_df = pd.read_csv(temp_output)
        
        # Merge with original data
        original_data = pd.DataFrame({
            'unique_id': [e[0] for e in valid_entries],
            'original_sequence': [e[1] for e in valid_entries],
            'original_site': [e[2] for e in valid_entries]})
        
        merged = original_data.merge(
            results_df,
            left_on=['unique_id', 'original_site'],
            right_on=['prot_id', 'position'],
            how='left')
        
        # Cleanup columns
        result_cols = ['unique_id', 'pred']
        return merged[result_cols].rename(columns={'pred': 'phospho_score'})
        
    finally:
        # Cleanup temporary files
        for f in [temp_fasta, temp_output]:
            if f.exists():
                f.unlink()

In [19]:
model_loc = "/Users/newuser/PhosphoLingo_ST_new.ckpt" 
# model_loc = r"H:\PhosphoLingo_ST_new.ckpt"
sequence_ids = ["test1"]
sequences = ["AsAAAARASKKKKKK"]
sites = [2]
extract_phospholingo_score(sequences, sites, model_loc)

As@AAAARASKKKKKK
LOADING MODEL NOW


/Users/newuser/anaconda3/envs/1433predictor2026/lib/python3.9/site-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at Rostlab/prot_t5_xl_uniref50 were not used when initializing T5EncoderModel: ['decoder.block.0.layer.1.EncDecAttention.v.weight', 'decoder.block.2.layer.2.layer_norm.weight', 'decoder.block.22.layer.0.SelfAttention.v.weight', 'decoder.block.13.layer.1.EncDecAttention.o.weight', 'decoder.block.22.layer.1.EncDecAttention.v.weight', 'decoder.block.1.layer.1.EncDecAttention.v.weight', 'decoder.block.20.layer.0.SelfAttention.v.weight', 'decoder.block.23.layer.1.EncDecAttention.v.weight', 'decoder.block.23.layer.0.SelfAttention.q.weight', 'decoder.block.20.layer.1.EncDecAttention.q.weight', 'decoder.block.15.layer.1.EncDecAttention.k.weight

--- Loaded temp_8106659b332b4b84b5f131cd3857116f.fasta data ---
- Number of proteins: 1
- Number of positive sites: 0
- Number of negative sites: 1



,unique_id,phospho_score
0,default,0.34


In [20]:
import gc

# Force garbage collection
gc.collect()

447

In [21]:
extract_phospholingo_score(sequences, sites, model_loc)

As@AAAARASKKKKKK
LOADING MODEL NOW


/Users/newuser/anaconda3/envs/1433predictor2026/lib/python3.9/site-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at Rostlab/prot_t5_xl_uniref50 were not used when initializing T5EncoderModel: ['decoder.block.0.layer.1.EncDecAttention.v.weight', 'decoder.block.2.layer.2.layer_norm.weight', 'decoder.block.22.layer.0.SelfAttention.v.weight', 'decoder.block.13.layer.1.EncDecAttention.o.weight', 'decoder.block.22.layer.1.EncDecAttention.v.weight', 'decoder.block.1.layer.1.EncDecAttention.v.weight', 'decoder.block.20.layer.0.SelfAttention.v.weight', 'decoder.block.23.layer.1.EncDecAttention.v.weight', 'decoder.block.23.layer.0.SelfAttention.q.weight', 'decoder.block.20.layer.1.EncDecAttention.q.weight', 'decoder.block.15.layer.1.EncDecAttention.k.weight

--- Loaded temp_6aa7fc1398954e13abfa138903a24067.fasta data ---
- Number of proteins: 1
- Number of positive sites: 0
- Number of negative sites: 1



,unique_id,phospho_score
0,default,0.34


### Extract ESM2 embedding functions

In [22]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel

# Load the ESM-2 model and tokenizer once (outside the function)
model_name = "facebook/esm2_t33_650M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def extract_esm_embedding(seq, site, tokenizer, model):
    """
    Extract protein-level and site-specific embeddings using ESM-2.
    
    Args:
        seq (str): Protein sequence.
        site (int): site of the protein for extrac embeddings.
        tokenizer: Pre-trained tokenizer.
        model: Pre-trained ESM-2 model.
    
    Returns:
        protein_embedding_df (pd.DataFrame): Protein-level embedding.
        site_embedding_df (pd.DataFrame): Site-specific embedding.
    """
    # Tokenize the sequence
    inputs = tokenizer(seq, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}  # Move inputs to GPU
    
    # Get embeddings
    with torch.no_grad():  # Disable gradient calculation
        outputs = model(**inputs, output_hidden_states=True)
    
    # Get residue-level embeddings (last layer)
    residue_embeddings = outputs.last_hidden_state[0, 1:-1]  # Remove special tokens

    # change protein site into site index
    site_index = site-1
    
    # Ensure the site is within the sequence length
    if site_index < 0 or site_index > len(seq):
        raise ValueError("Invalid site index. It must be within the range of the sequence length.")
    
    # Get the site-specific embedding
    site_embedding = residue_embeddings[site_index]
    
    # Get protein-level embedding (mean pooling)
    protein_embedding = torch.mean(residue_embeddings, dim=0)
    
    # Convert to DataFrames with dynamic column names
    protein_embedding_df = pd.DataFrame([protein_embedding.cpu().numpy()],
                                        columns=[f"esm_protein_embedding_{i}" for i in range(protein_embedding.shape[0])])
    site_embedding_df = pd.DataFrame([site_embedding.cpu().numpy()],
                                     columns=[f"esm_residue_embedding_{i}" for i in range(site_embedding.shape[0])])
    
    return protein_embedding_df, site_embedding_df

/Users/newuser/anaconda3/envs/1433predictor2026/lib/python3.9/site-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at facebook/esm2_t33_650M_UR50D were not used when initializing EsmModel: ['lm_head.dense.weight', 'lm_head.layer_norm.weight', 'lm_head.layer_norm.bias', 'esm.contact_head.regression.weight', 'esm.contact_head.regression.bias', 'lm_head.bias', 'lm_head.dense.bias', 'lm_head.decoder.weight']
- This IS expected if you are initializing EsmModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing EsmModel from the checkpoint of a model that you expect to be exactly identic

In [23]:
# List of protein sequences
protein_sequences = "SKKKTLSEAIAQIQES",

site = 1

protein_embedding_df, site_embedding_df = extract_esm_embedding(protein_sequences, site, tokenizer, model)

import gc
gc.collect() 

9

In [24]:
site_embedding_df

,esm_residue_embedding_0,esm_residue_embedding_1,esm_residue_embedding_2,esm_residue_embedding_3,esm_residue_embedding_4,esm_residue_embedding_5,esm_residue_embedding_6,esm_residue_embedding_7,esm_residue_embedding_8,esm_residue_embedding_9,...,esm_residue_embedding_1270,esm_residue_embedding_1271,esm_residue_embedding_1272,esm_residue_embedding_1273,esm_residue_embedding_1274,esm_residue_embedding_1275,esm_residue_embedding_1276,esm_residue_embedding_1277,esm_residue_embedding_1278,esm_residue_embedding_1279
0,0.096351,0.147667,-0.154439,0.102698,0.058262,0.003198,-0.023584,0.096275,0.189147,-0.195041,...,-0.073852,-0.031787,-0.083991,0.431811,-0.028847,-0.022237,-0.153851,0.057205,-0.169088,-0.161873


In [25]:
protein_embedding_df

,esm_protein_embedding_0,esm_protein_embedding_1,esm_protein_embedding_2,esm_protein_embedding_3,esm_protein_embedding_4,esm_protein_embedding_5,esm_protein_embedding_6,esm_protein_embedding_7,esm_protein_embedding_8,esm_protein_embedding_9,...,esm_protein_embedding_1270,esm_protein_embedding_1271,esm_protein_embedding_1272,esm_protein_embedding_1273,esm_protein_embedding_1274,esm_protein_embedding_1275,esm_protein_embedding_1276,esm_protein_embedding_1277,esm_protein_embedding_1278,esm_protein_embedding_1279
0,-0.007556,0.049159,-0.037948,0.100012,0.000864,-0.078121,-0.074609,0.026483,0.101897,-0.109056,...,0.056542,-0.011877,-0.113599,0.113303,0.003155,0.067324,0.031904,-0.005103,-0.062376,-0.026006


### Extract deephase score functions

In [26]:
# import pandas as pd

# #DO NOT CHANGE ANYTHING IN THIS CELL. MOVE ON TO THE FOLLOWING ONE TO GET THE PREDICTION.
# import os
# os.environ['PATH'] = "/Users/newuser/ncbi-blast-2.16.0+/bin:" + os.environ['PATH']
# os.environ['PATH'] = "C:/Program Files/NCBI/blast-2.16.0+/bin" + ";" + os.environ['PATH']


# import numpy as np
# from gensim.models import word2vec

# class ProtVec(word2vec.Word2Vec):

#     def __init__(self, fasta_fname=None, corpus=None, n=3, size=100, corpus_fname="corpus.txt",  sg=1, window=25, min_count=1, workers=20):
#         """
#         Either fname or corpus is required.
#         fasta_fname: fasta file for corpus
#         corpus: corpus object implemented by gensim
#         n: n of n-gram
#         corpus_fname: corpus file path
#         min_count: least appearance count in corpus. if the n-gram appear k times which is below min_count, the model does not remember the n-gram
#         """

#         self.n = n
#         self.size = size
#         self.fasta_fname = fasta_fname

#         if corpus is None and fasta_fname is None:
#             raise Exception("Either fasta_fname or corpus is needed!")

#         if fasta_fname is not None:
#             print('Generate Corpus file from fasta file...')
#             generate_corpusfile(fasta_fname, n, corpus_fname)
#             corpus = word2vec.Text8Corpus(corpus_fname)

#         word2vec.Word2Vec.__init__(self, corpus, size=size, sg=sg, window=window, min_count=min_count, workers=workers)

#     def to_vecs(self, seq):
#         """
#         convert sequence to three n-length vectors
#         e.g. 'AGAMQSASM' => [ array([  ... * 100 ], array([  ... * 100 ], array([  ... * 100 ] ]
#         """
#         ngram_patterns = split_ngrams(seq, self.n)

#         protvecs = []
#         for ngrams in ngram_patterns:
#             ngram_vecs = []
#             for ngram in ngrams:
#                 try:
#                     ngram_vecs.append(self.wv[ngram])
#                 except:
#                     raise Exception("Model has never trained this n-gram: " + ngram)
#             protvecs.append(sum(ngram_vecs))
#         return protvecs
    
    
#     def get_vector(self, seq):
#         """
#         sum and normalize the three n-length vectors returned by self.to_vecs
#         """
#         #return normalize(sum(self.to_vecs(seq)))
#         return sum(self.to_vecs(seq))

    
# def load_protvec(model_fname):
#     return word2vec.Word2Vec.load(model_fname)

# pv = load_protvec('src/files/DeePhase/__PREDICT/tools/Embeddings/swissprot_size200_window25.model')

# SEED = 42
# np.random.seed(SEED)

# from src.files.DeePhase.__PREDICT.deephase_utils import *

# def extract_seq_deephase_score(seq):
#     # # Create a DataFrame with the input sequence
#     df = pd.DataFrame({'sequence_final': [seq]})
    
#     # Call the DeePhase function (assuming it returns a string)
#     deephase_result = DeePhase(df)

#     df_deephase_score = pd.DataFrame([deephase_result], columns=['deephase_phys_multi', 'deephase_w2v_multi', 'deephase_score'])

#     df_deephase_score = df_deephase_score.astype(float)

#     return df_deephase_score

In [27]:
# sequence = "HTWDAAAAAAAAAAAAGAWEWSIDTEAGGGRREQSQKPCSNGGPAAAGEGRVLPSPCFPWSTCQAAIHKVCRWQGCTRPALLAPSLATLKEHSYP"
# extract_seq_deephase_score(sequence)

### Extract PTM-mamba embedding functions

In [28]:
# # need to run under docker enviroment
# # can not run directly from here
# # setup PTM-mamba into the folder 1433predictor\src\files\ptm-mamba-main, run infer_1433.py for inference.

# # processing the sequence to fit for the requirement for ptmmamba under the folder 1433predictor\src\files\ptm-mamba-main
# # save the processed seq and index into a csv file name as: ptmmamba_seq_index.csv
# import pandas as pd
# import os

# # mac add path
# # os.environ['PATH'] = os.environ['PATH'] + ':/usr/local/bin/docker'

# # windows add path

# def modify_sequence_for_phosphorylation(seq, site):
#     site_index = site - 1
#     print(seq)
#     if site_index < 0 or site_index > len(seq):
#         raise ValueError("Site index is out of range for the sequence length.")
#         return None
            
#     original_residue = seq[site_index]

#     if original_residue not in ["s", "t"]:
#         raise ValueError(f"Residue at site {site_index+1} is neither 's' nor 't'. It's '{original_residue}'.")
#         return None
        
#     if seq[-1] in ["s", "t"]:
#         seq += "A"
        
#     modified_seq = ""
#     for c in seq:    
#         if c == 's':
#             modified_residue = "<Phosphoserine>"
#             modified_seq += modified_residue
#         elif c == 't':
#             modified_residue = "<Phosphothreonine>"
#             modified_seq += modified_residue
#         else:
#             modified_seq += c
#     print(modified_seq)
#     return modified_seq

# def save_sequences_to_csv(sequences, sites, filename="src/files/ptm-mamba-main/ptmmamba_seq_index.csv"):
#     data = []
    
#     for seq, site in zip(sequences, sites):
#         try:
#             modified_sequence = modify_sequence_for_phosphorylation(seq, site)
#             data.append({"ptmmamba_seq": modified_sequence, "site_index": site})
#             print(data)
#         except ValueError as e:
#             print(f"Skipping sequence due to error: {e}")
#             continue

#     df = pd.DataFrame(data)
#     df.to_csv(filename, index=False)

# # function to running ptmmamba in docker
# # read the ptmmamba results into a df
# def extract_ptmmamba_embedding(sequence, site):    
#     save_sequences_to_csv([sequence], [site], filename="src/files/ptm-mamba-main/ptmmamba_seq_index.csv")
#     # save_sequences_to_csv([sequence], [site-1], filename="H:/ptm-mamba-main/ptmmamba_seq_index.csv")
    
#     # !docker start plm_benji && docker exec plm_benji python infer_1433.py && docker stop plm_benji
    
#     import subprocess

#     # Start the Docker container
#     start_container = subprocess.run(["docker", "start", "plm_benji"], check=True)

#     # Execute the Python script inside the Docker container
#     exec_script = subprocess.run(["docker", "exec", "plm_benji", "python", "infer_1433.py"], check=True)

#     # Stop the Docker container
#     # stop_container = subprocess.run(["docker", "stop", "plm_benji"], check=True)

#     # read the output or handle errors
#     try:
#         if start_container.returncode == 0 and exec_script.returncode == 0:
#             print("All commands executed successfully.")
#             try:
#                 df_temp = pd.read_csv("src/files/ptm-mamba-main/ptmmamba_embeddings.csv")
#                 phosphosite_embedding_df = df_temp.filter(like="ptmmamba_residue_embedding")
#                 phosphoprotein_embedding_df = df_temp.filter(like="ptmmamba_protein_embedding")
#             except Exception as e:
#                 print(f"Error reading CSV file: {e}")
#                 phosphosite_embedding_df = pd.DataFrame()  # Empty DataFrame instead of None
#                 phosphoprotein_embedding_df = pd.DataFrame()  # Empty DataFrame instead of None
#             finally:
#                 # Attempt to delete the CSV file regardless of read success
#                 try:
#                     os.remove("src/files/ptm-mamba-main/ptmmamba_embeddings.csv")
#                     print("CSV file deleted successfully.")
#                 except OSError as e:
#                     print(f"Failed to delete CSV file: {e}")
#         else:
#             print("An error occurred while executing the commands.")
#             phosphosite_embedding_df = pd.DataFrame()  # Empty DataFrame instead of None
#             phosphoprotein_embedding_df = pd.DataFrame()  # Empty DataFrame instead of None
#             # Even if commands failed, try to delete CSV if it exists
#             try:
#                 if os.path.exists("src/files/ptm-mamba-main/ptmmamba_embeddings.csv"):
#                     os.remove("src/files/ptm-mamba-main/ptmmamba_embeddings.csv")
#                     print("CSV file deleted successfully.")
#             except OSError as e:
#                 print(f"Failed to delete CSV file: {e}")
#     except Exception as e:
#         print(f"Unexpected error: {e}")
#         phosphosite_embedding_df = pd.DataFrame()  # Empty DataFrame instead of None
#         phosphoprotein_embedding_df = pd.DataFrame()  # Empty DataFrame instead of None
        
#     return phosphoprotein_embedding_df, phosphosite_embedding_df

In [29]:
# # test code
# sequence = "MsSQSHPDGLSGRDQPVELLNPARVNHMPSTVD"
# site = 2
# phosphoprotein_embedding_df, phosphosite_embedding_df = extract_ptmmamba_embedding(sequence, site)

In [30]:
# phosphosite_embedding_df

In [31]:
# phosphoprotein_embedding_df

In [32]:
# test code
# sequence = "St"
# site = 2
# phosphoprotein_embedding_df, phosphosite_embedding_df = extract_ptmmamba_embedding(sequence, site)

In [33]:
# phosphoprotein_embedding_df

In [34]:
# phosphosite_embedding_df

### Extract one_hot_encode functions

In [35]:
import numpy as np
import pandas as pd


def one_hot_encode(sequence):
    """
    One-hot encodes a protein sequence.

    Parameters:
        sequence (str): The protein sequence to be one-hot encoded.

    Returns:
        np.ndarray: A one-hot encoded representation of the sequence.
    """

    AMINO_ACIDS = 'ACDEFGHIKLMNPQRSTVWYst'
    one_hot = np.zeros((len(sequence), len(AMINO_ACIDS)), dtype=int)
    
    for i, char in enumerate(sequence):
        if char in AMINO_ACIDS:
            index = AMINO_ACIDS.index(char)
            one_hot[i, index] = 1
    
    return one_hot

def extract_onehot_embedding(sequence, site):
    """
    Extracts subsequences around specified phosphorylation sites and one-hot encodes them.

    Parameters:
        sequence (str): The protein sequence.
        site (int): The phosphorylation site (1-based index).

    Returns:
        pd.DataFrame: DataFrame with one-hot encoded features for the sequence window.
    """
    AMINO_ACIDS = 'ACDEFGHIKLMNPQRSTVWYst'
    data = {}  # Dictionary to hold data for new columns

    # Initialize a sequence of 15 "-" characters
    sub_sequence = ['-'] * 15

    # Convert to 0-based index
    site_index = site - 1
    
    # Determine the valid window around the phosphorylation site
    start = max(0, site_index - 7)
    end = min(len(sequence), site_index + 8)

    # Fill the sub_sequence with valid residues
    for i in range(start, end):
        sub_sequence[i - (site_index - 7)] = sequence[i]

    # One-hot encode the window
    one_hot = one_hot_encode(''.join(sub_sequence))

    # Create feature columns for each position and amino acid
    for pos_offset in range(-7, 8):
        position_in_onehot = pos_offset + 7  # Convert to 0-based index (0-14)
        for aa_index, aa in enumerate(AMINO_ACIDS):
            column_name = f"onehot_{pos_offset}_{aa}"
            data[column_name] = [one_hot[position_in_onehot, aa_index]]

    return pd.DataFrame(data)

In [36]:
# Example usage
sequence = "AsAAAAAAAAAAAAt"
site = 8

region_sequence = extract_onehot_embedding(sequence, site)
region_sequence

,onehot_-7_A,onehot_-7_C,onehot_-7_D,onehot_-7_E,onehot_-7_F,onehot_-7_G,onehot_-7_H,onehot_-7_I,onehot_-7_K,onehot_-7_L,...,onehot_7_P,onehot_7_Q,onehot_7_R,onehot_7_S,onehot_7_T,onehot_7_V,onehot_7_W,onehot_7_Y,onehot_7_s,onehot_7_t
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


### Extract idr sequence for extract idr features functions (include compactness score)

In [37]:
import os
import sys
import pandas as pd
import numpy as np
import src.files.iupred2a.iupred2a_lib as iupred2a_lib
from Bio import SeqIO

def extract_idr_seq(sequence, site):
    iupred_type = "long"
    # Predict disorder using iupred2a_lib
    iupred_scores = iupred2a_lib.iupred(sequence, iupred_type)[0]

    # Predict anchor regions using iupred2a_lib if anchor is enabled
    anchor_scores = iupred2a_lib.anchor2(sequence)[0]

    # Prepare the data for saving into a CSV file
    data = {
        "position": list(range(1, len(sequence) + 1)),
        "amino_acid": list(sequence),
        "iupred_score": iupred_scores,
        "anchor_score": anchor_scores,
    }

    # Create a DataFrame from the data
    df = pd.DataFrame(data)
    
    # Step 1: Add the 'call_idr' column
    df['call_idr'] = df['iupred_score'].apply(lambda x: 1 if x > 0.5 else 0)
    
    # Check if the provided site is within a '1' region
    if df.at[site-1, 'call_idr'] != 1:
        return None

    # Find the contiguous segments where call_idr == 1
    df['segment_id'] = (df['call_idr'] != df['call_idr'].shift()).cumsum()
    contiguous_segments = df[df['call_idr'] == 1].groupby('segment_id')
    
    for _, segment in contiguous_segments:
        # Check if the site falls within this segment
        if site in segment['position'].values:
            # Truncate the segment to a maximum length of 500 if necessary
            start_idx = segment.index[0]
            end_idx = segment.index[-1]

            if len(segment) > 500:
                # Ensure the site is within the truncated segment
                site_idx = df.index[df['position'] == site][0]
                if site_idx - start_idx > 250:
                    start_idx = max(site_idx - 250, start_idx)
                end_idx = min(start_idx + 500, end_idx)

            truncated_segment = df.loc[start_idx:end_idx]
            return "".join(truncated_segment['amino_acid'].tolist())

    return None

In [38]:
# Example usage
sequence = "MSSQSHPDGLSGRDQPVELLNPARVNHMPSTVDVATALPLQVAPSAVPMDLRLDHQFSLPVAEPALREQQLQQELLALKQKQQIQRQILIAEFQRQHEQLSRQHEAQLHEHIKQQQEMLAMKHQQELLEHQRKLERHRQEQELEKQHREQKLQQLKNKEKGKESAVASTEVKMKLQEFVLNKKKALAHRNLNHCISSDPRYWYGKTQHSSLDQSSPPQSGVSTSYNHPVLGMYDAKDDFPLRKTASEPNLKLRSRLKQKVAERRSSPLLRRKDGPVVTALKKRPLDVTDSACSSAPGSGPSSPNNSSGSVSAENGIAPAVPSIPAETSLAHRLVAREGSAAPLPLYTSPSLPNITLGLPATGPSAGTAGQQDAERLTLPALQQRLSLFPGTHLTPYLSTSPLERDGGAAHSPLLQHMVLLEQPPAQAPLVTGLGALPLHAQSLVGADRVSPSIHKLRQHRPLGRTQSAPLPQNAQALQHLVIQQQHQQFLEKHKQQFQQQQLQMNKIIPKPSEPARQPESHPEETEEELREHQALLDEPYLDRLPGQKEAHAQAGVQVKQEPIESDEEEAEPPREVEPGQRQPSEQELLFRQQALLLEQQRIHQLRNYQASMEAAGIPVSFGGHRPLSRAQSSPASATFPVSVQEPPTKPRFTTGLVYDTLMLKHQCTCGSSSSHPEHAGRIQSIWSRLQETGLRGKCECIRGRKATLEELQTVHSEAHTLLYGTNPLNRQKLDSKKLLGSLASVFVRLPCGGVGVDSDTIWNEVHSAGAARLAVGCVVELVFKVATGELKNGFAVVRPPGHHAEESTPMGFCYFNSVAVAAKLLQQRLSVSKILIVDWDVHHGNGTQQAFYSDPSVLYMSLHRYDDGNFFPGSGAPDEVGTGPGVGFNVNMAFTGGLDPPMGDAEYLAAFRTVVMPIASEFAPDVVLVSSGFDAVEGHPTPLGGYNLSARCFGYLTKQLMGLAGGRIVLALEGGHDLTAICDASEACVSALLGNELDPLPEKVLQQRPNANAVRSMEKVMEIHSKYWRCLQRTTSTAGRSLIEAQTCENEEAETVTAMASLSVGVKPAEKRPDEEPMEEEPPL"
site = 246

region_sequence = extract_idr_seq(sequence, site)
region_sequence

'MYDAKDDFPLRKTASEPNLKLRSRLKQKVAERRSSPLLRRKDGPVVTALKKRP'

In [39]:
def filter_sequence(seq):
    # Define standard amino acids
    standard_amino_acids = set("ACDEFGHIKLMNPQRSTVWYst")

    
    # Replace non-standard amino acids with 'A'
    filtered_seq = ''.join(c if c in standard_amino_acids else 'A' for c in seq)
    
    return filtered_seq

In [40]:
import subprocess
import platform

def is_docker_running():
    try:
        # Run a simple Docker command to check if it's responding
        subprocess.run(["docker", "info"], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        return True
    except subprocess.CalledProcessError:
        return False
    except FileNotFoundError:
        # 'docker' command not found, Docker is not installed
        return False

def start_docker():
    system = platform.system()

    if system == "Linux":
        # On most Linux systems, use systemctl to start Docker
        try:
            subprocess.run(["sudo", "systemctl", "start", "docker"], check=True)
            print("Docker started on Linux.")
        except subprocess.CalledProcessError as e:
            print(f"Failed to start Docker on Linux: {e}")

    elif system == "Darwin":
        # On macOS, use the Docker Desktop.app
        try:
            subprocess.run(["open", "-a", "Docker"], check=True)
            print("Docker started on macOS.")
        except subprocess.CalledProcessError as e:
            print(f"Failed to start Docker on macOS: {e}")

    elif system == "Windows":
        # On Windows, use PowerShell to start Docker Desktop
        try:
            subprocess.run(["powershell", "Start-Process", "-FilePath", '"C:\\Program Files\\Docker\\Docker\\Docker Desktop.exe"'], check=True)
            print("Docker started on Windows.")
        except subprocess.CalledProcessError as e:
            print(f"Failed to start Docker on Windows: {e}")

    else:
        print("Unsupported operating system for automatic Docker start")

In [41]:
import pandas as pd
import numpy as np
#@title <b>Preliminary operations</b>
import subprocess
# subprocess.run( 'pip install wget localcider==0.1.18'.split() )
# subprocess.run('pip uninstall scikit-learn -y'.split())
# subprocess.run('pip install scikit-learn==1.0.2'.split())
import itertools
from localcider.sequenceParameters import SequenceParameters
import wget
import os
from joblib import dump, load
# from google.colab import files
# from ipywidgets import IntProgress
from IPython.display import display
from IPython.display import clear_output
import argparse
import sys
import torch
from transformers import AutoTokenizer, AutoModel
import src.files.iupred2a.iupred2a_lib as iupred2a_lib
from Bio import SeqIO
# from src.files.utils import *

import pandas as pd

#DO NOT CHANGE ANYTHING IN THIS CELL. MOVE ON TO THE FOLLOWING ONE TO GET THE PREDICTION.
import os
os.environ['PATH'] = "/Users/newuser/ncbi-blast-2.16.0+/bin:" + os.environ['PATH']
os.environ['PATH'] = "C:/Program Files/NCBI/blast-2.16.0+/bin" + ";" + os.environ['PATH']


import numpy as np
from gensim.models import word2vec

class ProtVec(word2vec.Word2Vec):

    def __init__(self, fasta_fname=None, corpus=None, n=3, size=100, corpus_fname="corpus.txt",  sg=1, window=25, min_count=1, workers=20):
        """
        Either fname or corpus is required.
        fasta_fname: fasta file for corpus
        corpus: corpus object implemented by gensim
        n: n of n-gram
        corpus_fname: corpus file path
        min_count: least appearance count in corpus. if the n-gram appear k times which is below min_count, the model does not remember the n-gram
        """

        self.n = n
        self.size = size
        self.fasta_fname = fasta_fname

        if corpus is None and fasta_fname is None:
            raise Exception("Either fasta_fname or corpus is needed!")

        if fasta_fname is not None:
            print('Generate Corpus file from fasta file...')
            generate_corpusfile(fasta_fname, n, corpus_fname)
            corpus = word2vec.Text8Corpus(corpus_fname)

        word2vec.Word2Vec.__init__(self, corpus, size=size, sg=sg, window=window, min_count=min_count, workers=workers)

    def to_vecs(self, seq):
        """
        convert sequence to three n-length vectors
        e.g. 'AGAMQSASM' => [ array([  ... * 100 ], array([  ... * 100 ], array([  ... * 100 ] ]
        """
        ngram_patterns = split_ngrams(seq, self.n)

        protvecs = []
        for ngrams in ngram_patterns:
            ngram_vecs = []
            for ngram in ngrams:
                try:
                    ngram_vecs.append(self.wv[ngram])
                except:
                    raise Exception("Model has never trained this n-gram: " + ngram)
            protvecs.append(sum(ngram_vecs))
        return protvecs
    
    
    def get_vector(self, seq):
        """
        sum and normalize the three n-length vectors returned by self.to_vecs
        """
        #return normalize(sum(self.to_vecs(seq)))
        return sum(self.to_vecs(seq))

    
def load_protvec(model_fname):
    return word2vec.Word2Vec.load(model_fname)

pv = load_protvec('src/files/DeePhase/__PREDICT/tools/Embeddings/swissprot_size200_window25.model')

SEED = 42
np.random.seed(SEED)

from src.files.DeePhase.__PREDICT.deephase_utils import *

def extract_seq_deephase_score(seq):
    # # Create a DataFrame with the input sequence
    df = pd.DataFrame({'sequence_final': [seq]})
    
    # Call the DeePhase function (assuming it returns a string)
    deephase_result = DeePhase(df)

    df_deephase_score = pd.DataFrame([deephase_result], columns=['deephase_phys_multi', 'deephase_w2v_multi', 'deephase_score'])

    df_deephase_score = df_deephase_score.astype(float)

    return df_deephase_score

# build the features df:
def build_bio_features_df(sequence, site):
    
    wt_seq = filter_sequence(sequence) # sequence only have "ACDEFGHIKLMNPQRSTVWYst"
    print("filter_sequence: ", wt_seq)
    seq = wt_seq.upper()
    site = int(site)
        
    deephase_score_df = extract_seq_deephase_score(seq)
    # print("deephase_score_df:", deephase_score_df)
    
    idr_score_df = extract_idr_anchor_score(seq, site)
    # print("idr_score_df and achor_score_df:", idr_score_df)

    onehot_embedding_df = extract_onehot_embedding(wt_seq, site)
    # print("onehot_embedding_df:", onehot_embedding_df)
    
    idr_seq = extract_idr_seq(seq, site)
    # print("idr_seq:", idr_seq)

    if idr_seq is None or len(idr_seq) <= 3:
        data = {
        'nuSVR': 0.0,
        'SconfSVR/N (kB)': 0.0,
        'mean_lambda': 0.0,
        'SHD': 0.0,
        'SCD': 0.0,
        'kappa': 0.0,
        'FCR': 0.0,
        'NCPR': 0.0,
        'fK': 0.0,
        'fR': 0.0,
        'fE': 0.0,
        'fD': 0.0,
        'faro': 0.0
        }
        compactness_score_df = pd.DataFrame([data], index=['protein1'])
        # print("idr_seq is None, compactness_score_df:", compactness_score_df)        
    else:
        compactness_score_df = extract_compactness_score(idr_seq)
        # print("compactness_score_df:", compactness_score_df)

    
    # merge feature dfs
    deephase_score_df = deephase_score_df.reset_index(drop=True)
    idr_score_df = idr_score_df.reset_index(drop=True)
    onehot_embedding_df = onehot_embedding_df.reset_index(drop=True)
    compactness_score_df = compactness_score_df.reset_index(drop=True)
    
    bio_features_df = pd.concat([deephase_score_df, idr_score_df, onehot_embedding_df, compactness_score_df], axis=1)
    
    return bio_features_df


# Load the ESM-2 model and tokenizer once (outside the function)
model_name = "facebook/esm2_t33_650M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

import gc

def build_plm_features_df(sequence, site, tokenizer, model, phospholingo=True):

    # phospholingo model location
    model_loc = "/Users/newuser/PhosphoLingo_ST_new.ckpt" 
    # model_loc = "H:/PhosphoLingo_ST_new.ckpt"
    
    # # Move model to GPU if available
    # device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # model.to(device)

    # make sure docker is running
    if not is_docker_running():
        print("Docker is not running. Attempting to start Docker...")
        start_docker()
    else:
        print("Docker is already running.")

    
    wt_seq = filter_sequence(sequence) # sequence only have "ACDEFGHIKLMNPQRSTVWYst"
    print("filter_sequence: ", wt_seq)
    seq = wt_seq.upper()
    site = int(site)
    print(site)

    protein_embedding_df, site_embedding_df = extract_esm_embedding(seq, site, tokenizer, model)
    # print("protein_embedding_df, site_embedding_df:", protein_embedding_df, site_embedding_df)

    # phosphoprotein_embedding_df, phosphosite_embedding_df = extract_ptmmamba_embedding(wt_seq, site)
    # print("phosphoprotein_embedding_df, phosphosite_embedding_df:", phosphoprotein_embedding_df, phosphosite_embedding_df)

    gc.collect()
    
    if phospholingo:
        phospholingo_score_df = extract_phospholingo_score([wt_seq], [site], model_loc)
        # print("phospholingo_score_df:", phospholingo_score_df)    
        # merge feature dfs
        plm_features_df = pd.concat([protein_embedding_df, site_embedding_df, phospholingo_score_df], axis=1)
    else:
        plm_features_df = pd.concat([protein_embedding_df, site_embedding_df,], axis=1)
        

    gc.collect() 
    
    return plm_features_df

/Users/newuser/anaconda3/envs/1433predictor2026/lib/python3.9/site-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at facebook/esm2_t33_650M_UR50D were not used when initializing EsmModel: ['lm_head.dense.weight', 'lm_head.layer_norm.weight', 'lm_head.layer_norm.bias', 'esm.contact_head.regression.weight', 'esm.contact_head.regression.bias', 'lm_head.bias', 'lm_head.dense.bias', 'lm_head.decoder.weight']
- This IS expected if you are initializing EsmModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing EsmModel from the checkpoint of a model that you expect to be exactly identic

In [42]:
sequence = "MSSQSHPDGLSGRDQPVELLNPARVNHMPSTVDVATALPLQVAPSAVPMDLRLDHQFSLPVAEPALREQQLQQELLALKQKQQIQRQILIAEFQRQHEQLSRQHEAQLHEHIKQQQEMLAMKHQQELLEHQRKLERHRQEQELEKQHREQKLQQLKNKEKGKESAVASTEVKMKLQEFVLNKKKALAHRNLNHCISSDPRYWYGKTQHSSLDQSSPPQSGVSTSYNHPVLGMYDAKDDFPLRKTASEPNLKLRSRLKQKVAERRSSPLLRRKDGPVVTALKKRPLDVTDSACSSAPGSGPSSPNNSSGSVSAENGIAPAVPSIPAETSLAHRLVAREGSAAPLPLYTSPSLPNITLGLPATGPSAGTAGQQDAERLTLPALQQRLSLFPGTHLTPYLSTSPLERDGGAAHSPLLQHMVLLEQPPAQAPLVTGLGALPLHAQSLVGADRVSPSIHKLRQHRPLGRTQSAPLPQNAQALQHLVIQQQHQQFLEKHKQQFQQQQLQMNKIIPKPSEPARQPESHPEETEEELREHQALLDEPYLDRLPGQKEAHAQAGVQVKQEPIESDEEEAEPPREVEPGQRQPSEQELLFRQQALLLEQQRIHQLRNYQASMEAAGIPVSFGGHRPLSRAQSSPASATFPVSVQEPPTKPRFTTGLVYDTLMLKHQCTCGSSSSHPEHAGRIQSIWSRLQETGLRGKCECIRGRKATLEELQTVHSEAHTLLYGTNPLNRQKLDSKKLLGSLASVFVRLPCGGVGVDSDTIWNEVHSAGAARLAVGCVVELVFKVATGELKNGFAVVRPPGHHAEESTPMGFCYFNSVAVAAKLLQQRLSVSKILIVDWDVHHGNGTQQAFYSDPSVLYMSLHRYDDGNFFPGSGAPDEVGTGPGVGFNVNMAFTGGLDPPMGDAEYLAAFRTVVMPIASEFAPDVVLVSSGFDAVEGHPTPLGGYNLSARCFGYLTKQLMGLAGGRIVLALEGGHDLTAICDASEACVSALLGNELDPLPEKVLQQRPNANAVRSMEKVMEIHSKYWRCLQRTTSTAGRSLIEAQTCENEEAETVTAMASLSVGVKPAEKRPDEEPMEEEPPL"
site = 246

wt_seq = filter_sequence(sequence) # sequence only have "ACDEFGHIKLMNPQRSTVWYst"
seq = wt_seq.upper()
print(seq)

bio_features_df = build_bio_features_df(sequence, site)
bio_features_df

,deephase_phys_multi,deephase_w2v_multi,deephase_score,iupred_score,anchor_score,onehot_-7_A,onehot_-7_C,onehot_-7_D,onehot_-7_E,onehot_-7_F,...,SHD,SCD,kappa,FCR,NCPR,fK,fR,fE,fD,faro
0,0.261,0.692,0.476,0.618285,0.873385,0,0,0,0,1,...,2.95,1.695,0.167,0.434,0.17,0.151,0.151,0.038,0.075,0.038


In [43]:
print(seq)

MSSQSHPDGLSGRDQPVELLNPARVNHMPSTVDVATALPLQVAPSAVPMDLRLDHQFSLPVAEPALREQQLQQELLALKQKQQIQRQILIAEFQRQHEQLSRQHEAQLHEHIKQQQEMLAMKHQQELLEHQRKLERHRQEQELEKQHREQKLQQLKNKEKGKESAVASTEVKMKLQEFVLNKKKALAHRNLNHCISSDPRYWYGKTQHSSLDQSSPPQSGVSTSYNHPVLGMYDAKDDFPLRKTASEPNLKLRSRLKQKVAERRSSPLLRRKDGPVVTALKKRPLDVTDSACSSAPGSGPSSPNNSSGSVSAENGIAPAVPSIPAETSLAHRLVAREGSAAPLPLYTSPSLPNITLGLPATGPSAGTAGQQDAERLTLPALQQRLSLFPGTHLTPYLSTSPLERDGGAAHSPLLQHMVLLEQPPAQAPLVTGLGALPLHAQSLVGADRVSPSIHKLRQHRPLGRTQSAPLPQNAQALQHLVIQQQHQQFLEKHKQQFQQQQLQMNKIIPKPSEPARQPESHPEETEEELREHQALLDEPYLDRLPGQKEAHAQAGVQVKQEPIESDEEEAEPPREVEPGQRQPSEQELLFRQQALLLEQQRIHQLRNYQASMEAAGIPVSFGGHRPLSRAQSSPASATFPVSVQEPPTKPRFTTGLVYDTLMLKHQCTCGSSSSHPEHAGRIQSIWSRLQETGLRGKCECIRGRKATLEELQTVHSEAHTLLYGTNPLNRQKLDSKKLLGSLASVFVRLPCGGVGVDSDTIWNEVHSAGAARLAVGCVVELVFKVATGELKNGFAVVRPPGHHAEESTPMGFCYFNSVAVAAKLLQQRLSVSKILIVDWDVHHGNGTQQAFYSDPSVLYMSLHRYDDGNFFPGSGAPDEVGTGPGVGFNVNMAFTGGLDPPMGDAEYLAAFRTVVMPIASEFAPDVVLVSSGFDAVEGHPTPLGGYNLSARCFGYLTKQLMGLAGGRIVLALEGGHDLTAICDASEACVSALLGNELDPL

In [50]:
wt_seq = filter_sequence(sequence) # sequence only have "ACDEFGHIKLMNPQRSTVWYst"
seq = wt_seq.upper()
print(seq)

MSSQSHPDGLSGRDQPVELLNPARVNHMPSTVDVATALPLQVAPSAVPMDLRLDHQFSLPVAEPALREQQLQQELLALKQKQQIQRQILIAEFQRQHEQLSRQHEAQLHEHIKQQQEMLAMKHQQELLEHQRKLERHRQEQELEKQHREQKLQQLKNKEKGKESAVASTEVKMKLQEFVLNKKKALAHRNLNHCISSDPRYWYGKTQHSSLDQSSPPQSGVSTSYNHPVLGMYDAKDDFPLRKTASEPNLKLRSRLKQKVAERRSSPLLRRKDGPVVTALKKRPLDVTDSACSSAPGSGPSSPNNSSGSVSAENGIAPAVPSIPAETSLAHRLVAREGSAAPLPLYTSPSLPNITLGLPATGPSAGTAGQQDAERLTLPALQQRLSLFPGTHLTPYLSTSPLERDGGAAHSPLLQHMVLLEQPPAQAPLVTGLGALPLHAQSLVGADRVSPSIHKLRQHRPLGRTQSAPLPQNAQALQHLVIQQQHQQFLEKHKQQFQQQQLQMNKIIPKPSEPARQPESHPEETEEELREHQALLDEPYLDRLPGQKEAHAQAGVQVKQEPIESDEEEAEPPREVEPGQRQPSEQELLFRQQALLLEQQRIHQLRNYQASMEAAGIPVSFGGHRPLSRAQSSPASATFPVSVQEPPTKPRFTTGLVYDTLMLKHQCTCGSSSSHPEHAGRIQSIWSRLQETGLRGKCECIRGRKATLEELQTVHSEAHTLLYGTNPLNRQKLDSKKLLGSLASVFVRLPCGGVGVDSDTIWNEVHSAGAARLAVGCVVELVFKVATGELKNGFAVVRPPGHHAEESTPMGFCYFNSVAVAAKLLQQRLSVSKILIVDWDVHHGNGTQQAFYSDPSVLYMSLHRYDDGNFFPGSGAPDEVGTGPGVGFNVNMAFTGGLDPPMGDAEYLAAFRTVVMPIASEFAPDVVLVSSGFDAVEGHPTPLGGYNLSARCFGYLTKQLMGLAGGRIVLALEGGHDLTAICDASEACVSALLGNELDPL

In [ ]:
# web

In [51]:
def extract_features(sequence, site):
    site = int(site)

    # Create a new sequence with the letter at the specified site lowercased
    if 0 <= site-1 < len(sequence):
        new_sequence = sequence[:site-1] + sequence[site-1].lower() + sequence[site:]
        protein_sequences = new_sequence              
        
    # Process the new data
    bio_features_df = build_bio_features_df(protein_sequences, site)
    plm_features_df = build_plm_features_df(protein_sequences, site, tokenizer, model, phospholingo=True)
    
    bio_features_df = bio_features_df.reset_index(drop=True)
    print(bio_features_df)
    plm_features_df = plm_features_df.reset_index(drop=True)
    print(plm_features_df)

    result_df = pd.concat([bio_features_df, plm_features_df], axis=1)
    print(len(result_df))
    
    optimal_features = ['esm_residue_embedding_491', 'NCPR', 'onehot_-3_V', 'nuSVR', 'esm_residue_embedding_715', 'esm_residue_embedding_394', 'phospho_score', 'onehot_1_L', 'onehot_-2_G', 'onehot_1_R', 'onehot_-3_A', 'esm_residue_embedding_356', 'esm_residue_embedding_81', 'esm_residue_embedding_453', 'onehot_-3_E', 'iupred_score', 'onehot_1_P', 'onehot_-2_S', 'esm_residue_embedding_1171', 'onehot_-1_K', 'esm_residue_embedding_1242', 'esm_residue_embedding_53', 'onehot_7_G', 'onehot_-1_H', 'onehot_3_D', 'deephase_w2v_multi', 'onehot_-4_R', 'onehot_2_P', 'esm_residue_embedding_385', 'esm_residue_embedding_844', 'esm_residue_embedding_197', 'esm_residue_embedding_1215', 'esm_residue_embedding_1041', 'esm_residue_embedding_1074', 'onehot_2_K', 'esm_protein_embedding_68', 'esm_residue_embedding_482', 'onehot_-5_R', 'esm_residue_embedding_546', 'esm_residue_embedding_327', 'onehot_-2_R', 'onehot_-3_R', 'onehot_-3_G', 'esm_residue_embedding_1259']

    result_df = result_df[optimal_features]

    def extract_number(x):
        try:
            if isinstance(x, (int, float)):
                return x
            if isinstance(x, str) and x.startswith('[') and x.endswith(']'):
                x = ast.literal_eval(x)
            if isinstance(x, list) and len(x) == 1 and isinstance(x[0], (int, float)):
                return x[0]
        except:
            pass
        return x  # fallback if nothing works

    for col in result_df.columns:
        if result_df[col].dtype == 'object':
            result_df[col] = result_df[col].apply(extract_number)

    print(len(result_df))
    print(result_df)

    return result_df

In [52]:
# extract_features(sequence, site)

In [53]:
import joblib
predict_model = joblib.load('model/1433model_20260223.pkl')

def predict_score(sequence, site):
    # Get both inputs from form
    
    # Feature extraction using both inputs
    prediction_features = extract_features(sequence, site)  # Updated to accept site
    # check_non_numeric_values(prediction_features)
    print(prediction_features)
    
    # Prediction
    prediction = predict_model.predict(prediction_features)[0]
    probability = predict_model.predict_proba(prediction_features)[0]

    print(prediction, probability)

In [54]:
def check_non_numeric_values(df):
    # Initialize a flag to track if any non-numeric values were found
    found_non_numeric = False
    
    # Loop through each column in the DataFrame
    for column in df.columns:
        # Check if the column is numeric
        if pd.api.types.is_numeric_dtype(df[column]):
            # For numeric columns, check for NaN, inf, or other non-finite values
            non_finite_mask = ~np.isfinite(df[column])
            if non_finite_mask.any():
                found_non_numeric = True
                non_finite_indices = df.index[non_finite_mask].tolist()
                for idx in non_finite_indices:
                    print(f"Column '{column}' at index {idx}: Non-finite value {df.loc[idx, column]}")
        else:
            # For non-numeric columns, report all values
            found_non_numeric = True
            print(f"Column '{column}' is not numeric. Data type: {df[column].dtype}")
            # Show sample of non-numeric values
            unique_values = df[column].unique()
            print(f"Sample values: {unique_values[:5]}")
            
    # Check for specific non-numeric values in each cell
    for column in df.columns:
        # Try to convert column to numeric to catch non-convertible values
        numeric_series = pd.to_numeric(df[column], errors='coerce')
        # Find where NaN values were introduced by the conversion
        non_numeric_mask = numeric_series.isna() & ~df[column].isna()
        if non_numeric_mask.any():
            found_non_numeric = True
            non_numeric_indices = df.index[non_numeric_mask].tolist()
            for idx in non_numeric_indices:
                print(f"Column '{column}' at index {idx}: Non-numeric value '{df.loc[idx, column]}'")
    
    if not found_non_numeric:
        print("All values in the DataFrame are valid numbers.")
    
    return not found_non_numeric  # Return True if all values are numeric

In [55]:
predict_score(sequence, site)

Docker is already running.
filter_sequence:  MSSQSHPDGLSGRDQPVELLNPARVNHMPSTVDVATALPLQVAPSAVPMDLRLDHQFSLPVAEPALREQQLQQELLALKQKQQIQRQILIAEFQRQHEQLSRQHEAQLHEHIKQQQEMLAMKHQQELLEHQRKLERHRQEQELEKQHREQKLQQLKNKEKGKESAVASTEVKMKLQEFVLNKKKALAHRNLNHCISSDPRYWYGKTQHSSLDQSSPPQSGVSTSYNHPVLGMYDAKDDFPLRKTAsEPNLKLRSRLKQKVAERRSSPLLRRKDGPVVTALKKRPLDVTDSACSSAPGSGPSSPNNSSGSVSAENGIAPAVPSIPAETSLAHRLVAREGSAAPLPLYTSPSLPNITLGLPATGPSAGTAGQQDAERLTLPALQQRLSLFPGTHLTPYLSTSPLERDGGAAHSPLLQHMVLLEQPPAQAPLVTGLGALPLHAQSLVGADRVSPSIHKLRQHRPLGRTQSAPLPQNAQALQHLVIQQQHQQFLEKHKQQFQQQQLQMNKIIPKPSEPARQPESHPEETEEELREHQALLDEPYLDRLPGQKEAHAQAGVQVKQEPIESDEEEAEPPREVEPGQRQPSEQELLFRQQALLLEQQRIHQLRNYQASMEAAGIPVSFGGHRPLSRAQSSPASATFPVSVQEPPTKPRFTTGLVYDTLMLKHQCTCGSSSSHPEHAGRIQSIWSRLQETGLRGKCECIRGRKATLEELQTVHSEAHTLLYGTNPLNRQKLDSKKLLGSLASVFVRLPCGGVGVDSDTIWNEVHSAGAARLAVGCVVELVFKVATGELKNGFAVVRPPGHHAEESTPMGFCYFNSVAVAAKLLQQRLSVSKILIVDWDVHHGNGTQQAFYSDPSVLYMSLHRYDDGNFFPGSGAPDEVGTGPGVGFNVNMAFTGGLDPPMGDAEYLAAFRTVVMPIASEFAPDVVLVSSGFDAVEGHPTPLGGYNLSARCFGY

/Users/newuser/anaconda3/envs/1433predictor2026/lib/python3.9/site-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at Rostlab/prot_t5_xl_uniref50 were not used when initializing T5EncoderModel: ['decoder.block.0.layer.1.EncDecAttention.v.weight', 'decoder.block.2.layer.2.layer_norm.weight', 'decoder.block.22.layer.0.SelfAttention.v.weight', 'decoder.block.13.layer.1.EncDecAttention.o.weight', 'decoder.block.22.layer.1.EncDecAttention.v.weight', 'decoder.block.1.layer.1.EncDecAttention.v.weight', 'decoder.block.20.layer.0.SelfAttention.v.weight', 'decoder.block.23.layer.1.EncDecAttention.v.weight', 'decoder.block.23.layer.0.SelfAttention.q.weight', 'decoder.block.20.layer.1.EncDecAttention.q.weight', 'decoder.block.15.layer.1.EncDecAttention.k.weight

--- Loaded temp_1fe398c869d64a4dbf995e6046754aec.fasta data ---
- Number of proteins: 1
- Number of positive sites: 0
- Number of negative sites: 1

   deephase_phys_multi  deephase_w2v_multi  deephase_score  iupred_score  anchor_score  onehot_-7_A  onehot_-7_C  onehot_-7_D  onehot_-7_E  onehot_-7_F  ...   SHD    SCD  kappa    FCR  NCPR     fK     fR     fE     fD   faro
0  0.261                0.692               0.476           0.618285      0.873385      0            0            0            0            1            ...  2.95  1.695  0.167  0.434  0.17  0.151  0.151  0.038  0.075  0.038

[1 rows x 348 columns]
   esm_protein_embedding_0  esm_protein_embedding_1  esm_protein_embedding_2  esm_protein_embedding_3  esm_protein_embedding_4  esm_protein_embedding_5  esm_protein_embedding_6  esm_protein_embedding_7  esm_protein_embedding_8  esm_protein_embedding_9  ...  esm_residue_embedding_1272  esm_residue_embedding_1273  esm_residue_embedding_1274  esm_residue_embedding_1275  esm_res

In [56]:
import sys, torch, transformers
print(sys.executable)
print(sys.version)
print(torch.__version__)
print(transformers.__version__)

/Users/newuser/anaconda3/envs/1433predictor2026/bin/python
3.9.21 | packaged by conda-forge | (main, Dec  5 2024, 13:50:36) 
[Clang 18.1.8 ]
1.13.1
4.25.1
